# Calvin Site 1 crown and diameter search

Review the excavation priors, fitted settings and the fresh geometry comparisons. The entrypoint is the shared schema-4 generator.

In [ ]:
import json
import os
import subprocess
from hashlib import sha256
from pathlib import Path
from textwrap import dedent
import pandas as pd
from IPython.display import Image, display

project = next(root for root in (Path.cwd(), *Path.cwd().parents)
               if (root / '.runtime/calvin-site1-fit-20261119/crown20/PROTOCOL.json').is_file())
suite = project / '.runtime/calvin-site1-fit-20261119'
bundle = suite / 'crown20'
receipt = json.loads((bundle / 'PROTOCOL.json').read_text())
for relative, digest in receipt['inputs_sha256'].items():
    assert sha256((project / relative).read_bytes()).hexdigest() == digest
priors = json.loads((bundle / 'CROWN_PRIORS.json').read_text())
for order in ('2', '3'):
    print('Recorded order', order, priors['order_diameter'][order]['linked_starts_mm'])
display(pd.read_csv(bundle / 'development_ranks.csv')[['name','spatial_score','rld_depth_tv','mass_depth_tv']])

In [ ]:
result = json.loads((bundle / 'SUMMARY.json').read_text())
scores = pd.read_csv(bundle / 'spatial_scores.csv')
columns = ['case', 'cutoff_mm', 'qc', 'matched_seeds', 'spatial_score', 'rld_depth_tv', 'mass_depth_tv']
display(scores[columns].round(3))
display(pd.read_csv(bundle / 'parameter_values.csv'))
display(Image(filename=str(bundle / 'held_core_profiles.png'), width=1010))

## Replay the parameter set

Default cells use the saved receipts. Set `RUN_EXAMPLE` to simulate the printed configuration and verify both cutoff projections. This runs one local simulation. Recompute later stages from the saved trials with `continue_crown.py` when `CONTINUATION.json` exists. Other searches use `crown_search.py`.

In [ ]:
RUN_EXAMPLE = False
if RUN_EXAMPLE:
    source = project / '.runtime/spatial-construction-20261119/source'
    py310 = project / '.runtime/envs/py310-qualification/bin/python'
    env = dict(os.environ, PYTHONPATH=str(source), PYTHONDONTWRITEBYTECODE='1',
               OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1')
    program = dedent("""
        import sys
        import numpy as np
        sys.path.insert(0, sys.argv[1])
        import crown_search as crown
        study = crown.study
        crown.freeze(crown.read_inputs())
        receipt = study.cutoff.read(crown.OUT / 'fitted_input_replay.json')
        baseline = study.cutoff.read(crown.OUT / f"{receipt['name']}.schema4.private.json")
        _, _, tubes, offsets = study.observation_support()
        simulation = study.RootSystemSimulation(random_seed=baseline['random_seed'])
        simulation.run(baseline)
        for cutoff in study.cutoff.CUTOFFS:
            data = study.cutoff.observe(simulation, cutoff, tubes, offsets).set_index('target_row')[crown.SCORER.PREDICTIONS]
            np.testing.assert_array_equal(data.to_numpy(), receipt['predictions'][str(cutoff)])
        print('Both matched-cylinder projections reproduced.')
    """)
    subprocess.run([py310, '-c', program, suite], env=env, cwd=source, check=True)